<a href="https://colab.research.google.com/github/rahulpirwani7/Chest-Diease-Predication-With-CNN/blob/main/ChestDieasePredication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import os
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import kagglehub

In [ ]:

dataset_path = kagglehub.dataset_download(
    "nguynhongmaivy/dataset-nih-chestx-ray14"
)

print(dataset_path)

In [ ]:
train_path = os.path.join(dataset_path, "train.csv")
val_path = os.path.join(dataset_path, "val.csv")
test_path = os.path.join(dataset_path, "test.csv")

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

In [ ]:
image_paths = {}

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.lower().endswith((".png", ".jpg", ".jpeg")):
            image_paths[file] = os.path.join(root, file)

print("Total images found:", len(image_paths))

In [ ]:
train_df = train_df[
    train_df["Image Index"].isin(image_paths)
].reset_index(drop=True)

val_df = val_df[
    val_df["Image Index"].isin(image_paths)
].reset_index(drop=True)

test_df = test_df[
    test_df["Image Index"].isin(image_paths)
].reset_index(drop=True)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

In [ ]:
disease_columns = [
    "Atelectasis",
    "Cardiomegaly",
    "Effusion",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pneumonia",
    "Pneumothorax",
    "Consolidation",
    "Edema",
    "Emphysema",
    "Fibrosis",
    "Pleural_Thickening",
    "Hernia"
]

print(disease_columns)

In [ ]:
print(train_df.columns.tolist())

In [ ]:
train_df["label"] = (
    train_df[disease_columns].sum(axis=1) > 0
).astype("float32")

val_df["label"] = (
    val_df[disease_columns].sum(axis=1) > 0
).astype("float32")

test_df["label"] = (
    test_df[disease_columns].sum(axis=1) > 0
).astype("float32")

In [ ]:
print("TRAIN")
print(train_df["label"].value_counts())

print("\nVALIDATION")
print(val_df["label"].value_counts())

print("\nTEST")
print(test_df["label"].value_counts())

In [ ]:
train_df["image_path"] = train_df["Image Index"].map(image_paths)

val_df["image_path"] = val_df["Image Index"].map(image_paths)

test_df["image_path"] = test_df["Image Index"].map(image_paths)

In [ ]:
print("Train missing:", train_df["image_path"].isna().sum())
print("Validation missing:", val_df["image_path"].isna().sum())
print("Test missing:", test_df["image_path"].isna().sum())

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 64

In [ ]:
def load_image(path, label):
    image = tf.io.read_file(path)

    image = tf.image.decode_png(
        image,
        channels=3
    )

    image = tf.image.resize(
        image,
        IMG_SIZE
    )

    image = tf.cast(
        image,
        tf.float32
    ) / 255.0

    return image, label

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomTranslation(0.05, 0.05)
])

In [ ]:
def augment(image, label):
    image = data_augmentation(
        image,
        training=True
    )

    return image, label

In [ ]:
def create_dataset(df, augment_data=False):

    ds = tf.data.Dataset.from_tensor_slices(
        (
            df["image_path"].values,
            df["label"].values
        )
    )

    # Load images
    ds = ds.map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # Augment only training data
    if augment_data:
        ds = ds.map(
            augment,
            num_parallel_calls=tf.data.AUTOTUNE
        )

        ds = ds.shuffle(1000)

    # Batch
    ds = ds.batch(BATCH_SIZE)

    # Prefetch
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

In [ ]:
train_ds = create_dataset(
    train_df,
    augment_data=True
)

print("Train dataset ready!")

In [ ]:
val_ds = create_dataset(
    val_df,
    augment_data=False
)

print("Validation dataset ready!")

In [ ]:
test_ds = create_dataset(
    test_df,
    augment_data=False
)

print("Test dataset ready!")

In [ ]:
for images, labels in train_ds.take(1):

    print("Images shape:", images.shape)
    print("Labels shape:", labels.shape)

    print("First 10 labels:")
    print(labels[:10].numpy())

In [ ]:
import matplotlib.pyplot as plt

# Get one batch from train_ds
for images, labels in train_ds.take(1):
    image = images[0]
    label = labels[0]

print("Image shape:", image.shape)
print("Label:", label.numpy())

# Display 6 augmented versions
plt.figure(figsize=(12, 8))

for i in range(6):

    # Apply augmentation directly
    augmented = data_augmentation(
        image,
        training=True
    )

    plt.subplot(2, 3, i + 1)
    plt.imshow(augmented)
    plt.title(f"Augmented {i + 1}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import tensorflow as tf

# Create a new CNN from scratch
model = tf.keras.Sequential([

    # Input
    tf.keras.layers.Input(shape=(224, 224, 3)),

    # =========================
    # Block 1
    # =========================
    tf.keras.layers.Conv2D(
        32,
        (3, 3),
        padding="same",
        use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    # =========================
    # Block 2
    # =========================
    tf.keras.layers.Conv2D(
        64,
        (3, 3),
        padding="same",
        use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    # =========================
    # Block 3
    # =========================
    tf.keras.layers.Conv2D(
        128,
        (3, 3),
        padding="same",
        use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    # =========================
    # Block 4
    # =========================
    tf.keras.layers.Conv2D(
        256,
        (3, 3),
        padding="same",
        use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    # =========================
    # Classifier
    # =========================
    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(
        128,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.4),

    # Binary classification
    tf.keras.layers.Dense(
        1,
        activation="sigmoid"
    )
])

# Show architecture
model.summary()

# Compile
model.compile(
    optimizer=tf.keras.optimizers.SGD(
        learning_rate=0.01,
        momentum=0.9
    ),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc")
    ]
)

print("Model compiled successfully!")

In [ ]:
import tensorflow as tf

model = tf.keras.Sequential([

    tf.keras.layers.Input(shape=(224, 224, 3)),

    # =========================================================
    # STEM
    # =========================================================
    tf.keras.layers.Conv2D(
        32, 3, padding="same", use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.Conv2D(
        32, 3, padding="same", use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.MaxPooling2D(pool_size=2),

    # =========================================================
    # BLOCK 1
    # =========================================================
    tf.keras.layers.Conv2D(
        64, 3, padding="same", use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.Conv2D(
        64, 3, padding="same", use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.MaxPooling2D(pool_size=2),
    tf.keras.layers.Dropout(0.15),

    # =========================================================
    # BLOCK 2
    # =========================================================
    tf.keras.layers.Conv2D(
        128, 3, padding="same", use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.Conv2D(
        128, 3, padding="same", use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.Conv2D(
        128, 3, padding="same", use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.MaxPooling2D(pool_size=2),
    tf.keras.layers.Dropout(0.20),

    # =========================================================
    # BLOCK 3
    # =========================================================
    tf.keras.layers.Conv2D(
        256, 3, padding="same", use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.Conv2D(
        256, 3, padding="same", use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.Conv2D(
        256, 3, padding="same", use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.MaxPooling2D(pool_size=2),
    tf.keras.layers.Dropout(0.25),

    # =========================================================
    # BLOCK 4
    # =========================================================
    tf.keras.layers.Conv2D(
        512, 3, padding="same", use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.Conv2D(
        512, 3, padding="same", use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.Conv2D(
        512, 3, padding="same", use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    # =========================================================
    # FEATURE AGGREGATION
    # =========================================================
    tf.keras.layers.GlobalAveragePooling2D(),

    # =========================================================
    # CLASSIFIER
    # =========================================================
    tf.keras.layers.Dense(
        256,
        activation="relu"
    ),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(
        128,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.3),

    # Binary classification
    tf.keras.layers.Dense(
        1,
        activation="sigmoid"
    )
])

model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=tf.keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=0.01,
            decay_steps=1000,
            decay_rate=1,
            staircase=True
        )
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20
)

In [ ]:
print(train_df["label"].value_counts())

In [ ]:
test_loss, test_accuracy = model.evaluate(
    test_ds
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")

plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")

plt.legend()
plt.show()